In [1]:
!pip install nvcc4jupyter
%load_ext nvcc4jupyter

Detected platform "Colab". Running its setup...
Source files will be saved in "/tmp/tmp0h9za8c3".


In [7]:
%%writefile add.cu
#include <stdio.h>
#include <iostream>

__global__ void hello_kernel() {
    printf("Hello from thread %d, block %d\n", threadIdx.x, blockIdx.x);
}

__global__ void add(int n, float *x, float *y) {
    for (int i = 0; i < n; i++)
        y[i] = x[i] + y[i];
}

int main() {
    int N = 1<<20;
    float *x, *y;

    cudaMallocManaged(&x, N*sizeof(float));
    cudaMallocManaged(&y, N*sizeof(float));

    for (int i = 0; i < N; i++) {
        x[i] = 1.0f;
        y[i] = 2.0f;
    }

    hello_kernel<<<2,4>>>();

    add<<<1, 1>>>(N, x, y);
    cudaDeviceSynchronize();

    float maxError = 0.0f;
    for (int i = 0; i < N; i++)
        maxError = fmax(maxError, fabs(y[i] - 3.0f));
    std::cout << "Max error: " << maxError << std::endl;

    cudaFree(x);
    cudaFree(y);
    return 0;
}


Overwriting add.cu


In [8]:
!nvcc add.cu -o add_cuda && ./add_cuda

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
Hello from thread 0, block 1
Hello from thread 1, block 1
Hello from thread 2, block 1
Hello from thread 3, block 1
Hello from thread 0, block 0
Hello from thread 1, block 0
Hello from thread 2, block 0
Hello from thread 3, block 0
Max error: 0
